In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# 시간-주파수 분석: 단시간 푸리에 변환(STFT)

FFT 는 신호 전체를 하나의 스펙트럼으로 압축하므로 **언제 어떤 주파수가 나타났는지**를 알려 주지 않는다. 엔진음, 타건음, 충격성 이벤트처럼 시간에 따라 성분이 변하는 신호를 분석하려면, 신호를 짧은 구간으로 나누어 구간마다 FFT 를 적용하는 단시간 푸리에 변환(short-time Fourier transform, STFT)이 필요하다.

이 노트북에서는 엔진과 관련된 세 가지 음원(`autorash`, `knocking`, `f1music`)을 사용하여 다음을 차례로 관찰한다.

- 윈도우 길이(`nperseg`)가 시간 분해능과 주파수 분해능의 트레이드오프를 어떻게 결정하는지
- 이웃한 윈도우의 겹침(`noverlap`)이 시간 축 방향의 부드러움을 어떻게 좌우하는지
- Mel 주파수 축과 dB 크기가 청각적 해석에 주는 효과

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import utils
from scipy.signal import stft

try:
    import librosa
    import librosa.display
except ImportError:
    %pip install librosa
    import librosa
    import librosa.display

---


## 실습 1. 두 엔진음 훑어보기

- `autorash.mp3`: 오토래시(autorash) 결함이 있을 때 발생하는 엔진음으로, 회전수가 거의 일정하다.
- `knocking.mp3`: 엔진의 노킹음으로, 이상 연소로 인한 충격이 주기적으로 발생한다.

두 음원을 원래의 샘플링 주파수 그대로 불러온 뒤 길이를 맞추고, 시간 파형과 FFT 진폭 스펙트럼을 2 × 2 subplot 으로 나란히 배치하여 비교한다. 시간 축은 부동소수점 누적 오차가 없는 `np.arange(len(v))/fs` 로 만든다.

In [ ]:
# 데이터 로드 (원본 샘플링 그대로)
v_a, fs_a = librosa.load('./data/autorash.mp3',  sr=None)
v_k, fs_k = librosa.load('./data/knocking.mp3', sr=None)

# 두 신호 길이를 짧은 쪽 (knocking) 에 맞춤
N = min(len(v_a), len(v_k))
v_a = v_a[:N]
v_k = v_k[:N]

# 시간축: 부동소수점 누적 오차 방지 위해 index / fs 방식
t_a_raw = np.arange(len(v_a)) / fs_a
t_k_raw = np.arange(len(v_k)) / fs_k

print(f'Engine sound with autorash fault : shape={v_a.shape}, fs={fs_a} Hz, duration={len(v_a)/fs_a:.2f} s')
print(f'Knocking                         : shape={v_k.shape}, fs={fs_k} Hz, duration={len(v_k)/fs_k:.2f} s')

# 주파수 스펙트럼 (단측)
f_a, A_a = utils.fft(v_a, fs_a)
f_k, A_k = utils.fft(v_k, fs_k)

fig, ax = plt.subplots(2, 2, figsize=(12, 6))

ax[0, 0].plot(t_a_raw, v_a, 'C0')
ax[0, 0].set_title('Engine sound with autorash fault — Time')
ax[0, 0].set_xlabel('Time (s)'); ax[0, 0].set_ylabel('Amplitude')

ax[0, 1].plot(f_a, A_a, 'C0')
ax[0, 1].set_title('Engine sound with autorash fault — Spectrum')
ax[0, 1].set_xlabel('Frequency (Hz)'); ax[0, 1].set_ylabel('Magnitude')
ax[0, 1].set_xlim(0, 2000)

ax[1, 0].plot(t_k_raw, v_k, 'C1')
ax[1, 0].set_title('Knocking — Time')
ax[1, 0].set_xlabel('Time (s)'); ax[1, 0].set_ylabel('Amplitude')

ax[1, 1].plot(f_k, A_k, 'C1')
ax[1, 1].set_title('Knocking — Spectrum')
ax[1, 1].set_xlabel('Frequency (Hz)'); ax[1, 1].set_ylabel('Magnitude')
ax[1, 1].set_xlim(0, 2000)

fig.tight_layout(); plt.show()

**관찰.** 시간 파형에서 `knocking` 은 큰 충격성 스파이크 15개가 약 77 ms 간격으로, `autorash` 는 진폭이 커지는 구간 12개가 약 95 ms 간격으로 반복되는 것을 눈으로 확인할 수 있다. 그러나 각 사건이 어느 주파수 대역에서 발생하는지는 파형만으로 알 수 없다. 반대로 스펙트럼은 `autorash` 의 84, 127, 168 Hz 고조파 성분과 `knocking` 의 0 ~ 2 kHz 에 촘촘하게 분포하는 여러 성분을 보여 줄 뿐, **충격이 언제 발생했는지**는 전혀 알려 주지 않는다. `knocking` 충격의 에너지는 주로 4 kHz 위에 있으므로 가로축이 0 ~ 2 kHz 인 오른쪽 아래 패널의 범위 밖에 있다. 이 정보 손실을 보완하는 도구가 STFT 이다.

---


## 실습 2. 첫 스펙트로그램(`nperseg=256`, `noverlap=128`)

STFT 는 길이가 `nperseg` 인 윈도우마다 FFT 를 적용하며, 이웃한 윈도우가 `noverlap` 샘플만큼 겹치도록 윈도우를 시간 방향으로 이동한다.

- `nperseg = 256` 이면 윈도우 길이는 `256 / 44100 ≈ 5.8 ms` 이며, 충격성 이벤트를 시간적으로 분해하기에 적합하다.
- `noverlap = 128` 이면 겹침은 50 %이다.
- 주파수 분해능은 `fs / nperseg = 44100 / 256 ≈ 172 Hz` 로 비교적 낮다.

두 엔진음의 스펙트로그램을 그리며, STFT 가 반환하는 시간 축은 원 신호의 시간 축과 섞이지 않도록 `t_a_stft`, `t_k_stft` 라는 **별도 변수**에 저장한다.

In [ ]:
nperseg  = 256
noverlap = 128
nfft     = 256

f_a_stft, t_a_stft, Zxx_a = stft(v_a, fs=fs_a,
                                  nperseg=nperseg, noverlap=noverlap, nfft=nfft)
f_k_stft, t_k_stft, Zxx_k = stft(v_k, fs=fs_k,
                                  nperseg=nperseg, noverlap=noverlap, nfft=nfft)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

im0 = ax[0].pcolormesh(t_a_stft, f_a_stft, np.abs(Zxx_a),
                        shading='gouraud', cmap='viridis')
ax[0].set_title(f'Engine sound with autorash fault — STFT (nperseg={nperseg})')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('Frequency (Hz)')
ax[0].set_ylim(0, 10000)
fig.colorbar(im0, ax=ax[0], label='Magnitude')

im1 = ax[1].pcolormesh(t_k_stft, f_k_stft, np.abs(Zxx_k),
                        shading='gouraud', cmap='viridis')
ax[1].set_title(f'Knocking — STFT (nperseg={nperseg})')
ax[1].set_xlabel('Time (s)'); ax[1].set_ylabel('Frequency (Hz)')
ax[1].set_ylim(0, 10000)
fig.colorbar(im1, ax=ax[1], label='Magnitude')

fig.tight_layout(); plt.show()

**관찰.** `knocking` 에서는 짧은 광대역 충격에 의한 세로 줄무늬가 약 4 ~ 10 kHz 에 걸쳐 약 0.077 초마다 뚜렷하게 반복된다. 큰 충격 사이에 있는 작은 충격은 3 kHz 아래의 불규칙한 무늬에 섞여 있다. `autorash` 에서도 약 0.095 초마다 반복되는 무늬가 4 ~ 7 kHz 에 보이며, 이 무늬는 노킹의 세로 줄무늬보다 주파수 대역이 좁고 시간 폭이 넓다. 다만 두 패널은 컬러바가 따로 있으므로 두 무늬의 밝기는 서로 비교하지 않는다. FFT 한 장으로는 전혀 볼 수 없던 **시간적 구조**가 나타난다는 점이 중요하다. 그러나 주파수 분해능이 170 Hz 수준으로 낮으므로 가는 가로 줄무늬는 아직 보이지 않는다. 가로 줄무늬는 윈도우를 길게 잡으면 나타난다. knocking 의 가로 줄무늬는 실습 3에서 볼 수 있으며, 이 줄무늬는 충격음의 주파수 구성이 시간 축을 따라 이어진 것이다. autorash 는 `nperseg=8192` 로 직접 그려 본다.

---


## 실습 3. `nperseg` 비교

윈도우 길이를 바꾸면 시간 분해능과 주파수 분해능이 서로 반대로 변하며, 이 관계를 시간-주파수 불확정성이라고 한다. `knocking` 신호에 세 가지 값을 적용하여 결과를 나란히 비교한다.

| `nperseg` | 윈도우 길이 | 주파수 분해능 `fs/nperseg` | 특징 |
|-----------|--------------|----------------------------|------|
| 256       | ≈ 5.8 ms     | ≈ 172 Hz                   | 충격 이벤트의 시점이 뚜렷하고 주파수는 번진다 |
| 2048      | ≈ 46 ms      | ≈ 22 Hz                    | 두 분해능이 균형을 이룬다 |
| 8192      | ≈ 186 ms     | ≈ 5.4 Hz                   | 가로 줄무늬가 선명하고 시간 변화는 흐리다 |

겹침은 모두 50 %로 고정한다. 노킹 충격의 에너지는 약 4 ~ 10 kHz 에 있으므로 세로축은 실습 2와 같이 10 kHz 까지 그린다. 세로축을 5 kHz 아래로 자르면 가장 강한 충격 대역이 빠져 세로 줄무늬가 흐려진다.

In [ ]:
nperseg_list = [256, 2048, 8192]

fig, ax = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for i, nps in enumerate(nperseg_list):
    f_k_i, t_k_i, Zxx_k_i = stft(v_k, fs=fs_k,
                                  nperseg=nps, noverlap=nps // 2, nfft=nps)
    im = ax[i].pcolormesh(t_k_i, f_k_i, np.abs(Zxx_k_i),
                           shading='gouraud', cmap='viridis')
    ax[i].set_title(f'nperseg={nps}  (Δf ≈ {fs_k/nps:.1f} Hz, Δt ≈ {1000*nps/fs_k:.1f} ms)')
    ax[i].set_xlabel('Time (s)')
    ax[i].set_ylim(0, 10000)
    fig.colorbar(im, ax=ax[i], label='Magnitude')

ax[0].set_ylabel('Frequency (Hz)')
fig.suptitle('Knocking — STFT with varying nperseg (noverlap = nperseg/2)')
fig.tight_layout(); plt.show()

**관찰.** `nperseg=256` 에서는 노킹 충격마다 약 4 ~ 10 kHz 에 걸친 세로 줄무늬가 밀리초 단위로 날카롭게 나타나지만, 세로축 방향으로 색이 넓게 번지므로 가는 가로 줄무늬는 보이지 않는다. 반대로 `nperseg=8192` 에서는 윈도우 하나(약 186 ms)에 충격이 두세 개 들어가므로 개별 충격의 시점을 구분할 수 없다. 그 대신 어느 윈도우에서나 충격이 두세 개씩 섞여 거의 같은 스펙트럼이 얻어지므로, 시간 내내 이어지는 가로 줄무늬가 나타난다. 가로 줄무늬 가운데에서는 1.66 kHz 부근의 줄무늬가 가장 밝다. `nperseg=2048` 에서는 윈도우(약 46 ms)가 충격 간격(약 77 ms)보다 짧으므로 충격마다 시간 폭이 넓은 무늬가 따로 보이고, 그 안에 짧게 끊긴 가로 줄무늬가 함께 보인다. 충격과 그 주파수 구성을 함께 보려면 `nperseg=2048` 부근이 실용적인 절충점이다.

---


## 실습 4. `noverlap` 효과

`noverlap` 은 이웃한 윈도우가 공유하는 샘플의 수를 정한다. 겹침을 크게 하면 시간 축 방향의 프레임 수가 많아져 스펙트로그램이 부드럽게 보이지만, 계산량도 같은 비율로 증가한다.

`knocking` 신호에서 `nperseg = 2048` 로 고정하고 겹침이 50 %, 75 %, 95 %인 세 경우를 비교한다.

In [ ]:
nperseg = 2048
overlap_ratios = [0.5, 0.75, 0.95]

fig, ax = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for i, r in enumerate(overlap_ratios):
    nov = int(nperseg * r)
    f_k_i, t_k_i, Zxx_k_i = stft(v_k, fs=fs_k,
                                  nperseg=nperseg, noverlap=nov, nfft=nperseg)
    im = ax[i].pcolormesh(t_k_i, f_k_i, np.abs(Zxx_k_i),
                           shading='gouraud', cmap='viridis')
    ax[i].set_title(f'noverlap = {int(r*100)}%  ({len(t_k_i)} frames)')
    ax[i].set_xlabel('Time (s)')
    ax[i].set_ylim(0, 10000)
    fig.colorbar(im, ax=ax[i], label='Magnitude')

ax[0].set_ylabel('Frequency (Hz)')
fig.suptitle(f'Knocking — STFT with varying noverlap (nperseg={nperseg})')
fig.tight_layout(); plt.show()

**관찰.** 각 패널의 제목에 표시한 것처럼 겹침을 크게 할수록 시간 축의 프레임 수가 크게 늘고, 스펙트로그램이 눈으로 보기에 부드럽게 이어진다. 특히 50 % 겹침에서는 프레임 간격(약 23 ms)이 넓어 `knocking` 의 충격에 의한 무늬마다 위치와 밝기가 조금씩 불규칙하지만, 95 % 겹침(약 2.3 ms 간격)에서는 이 무늬들이 고른 간격과 밝기로 정리된다. 이 차이는 그림을 확대하면 잘 보인다. 물리적인 정보량이 늘어난 것은 아니며, 이 효과는 **시각적 보간**에 가깝다. 실무에서는 보통 50 ~ 75 %를 표준으로 사용하고, 발표 자료에 쓸 그림에서만 겹침을 더 크게 한다.

---


## 실습 5. F1 엔진음(`nperseg=2048`)

F1 엔진음은 여러 고조파 성분의 주파수가 함께 바뀌는 신호이다. 이 신호에서는 고조파의 구조를 읽는 것이 중요하므로, 고조파 사이의 간격을 분해할 만큼 길면서 주파수가 바뀌는 순간을 놓치지 않을 만큼 짧은 **중간 길이 윈도우**가 적합하다. 실습 3에서 두 분해능이 균형을 이룬 `nperseg=2048`(≈ 46 ms)을 50 % 겹침으로 사용한다.

In [ ]:
v_f1, fs_f1 = librosa.load('./data/f1music.mp3', sr=None)
t_f1_raw = np.arange(len(v_f1)) / fs_f1
print(f'F1 engine : shape={v_f1.shape}, fs={fs_f1} Hz, duration={len(v_f1)/fs_f1:.2f} s')

nperseg = 2048
noverlap = 1024
f_f1_stft, t_f1_stft, Zxx_f1 = stft(v_f1, fs=fs_f1,
                                     nperseg=nperseg, noverlap=noverlap)

fig, ax = plt.subplots(2, 1, figsize=(12, 8))

ax[0].plot(t_f1_raw, v_f1, 'C2')
ax[0].set_title('F1 Engine — Time')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('Amplitude')

im = ax[1].pcolormesh(t_f1_stft, f_f1_stft, np.abs(Zxx_f1),
                       shading='gouraud', cmap='viridis')
ax[1].set_title(f'F1 Engine — STFT (nperseg={nperseg}, noverlap={noverlap})')
ax[1].set_xlabel('Time (s)'); ax[1].set_ylabel('Frequency (Hz)')
ax[1].set_ylim(0, 1000)
fig.colorbar(im, ax=ax[1], label='Magnitude')

fig.tight_layout(); plt.show()

**관찰.** 여러 고조파의 줄무늬가 같은 주파수 비율을 유지하며 평평하게 이어지다가 **계단처럼 주파수가 바뀐다**. 8 ~ 9 초에는 줄무늬가 대각선으로 약 650 Hz 에서 500 Hz 까지 내려갔다가 9 초 무렵 짧은 시간 안에 다시 올라가 원래의 주파수로 돌아온다. 11 초 부근에서는 줄무늬가 잠깐 올라갔다가 내려온다. 회전수가 바뀌면 기본 주파수 `f0(t)` 와 그 정수배인 `2f0, 3f0, …` 가 함께 변한다는 점이 한 줄로 이루어진 chirp 와 다르다. `nperseg=2048` 의 주파수 분해능은 약 22 Hz 이므로 고조파 사이의 간격이 뚜렷하게 분리되며, 약 23 ms 의 프레임 간격으로 주파수가 바뀌는 순간을 따라갈 수 있다.

---


## 실습 6. Mel 스펙트로그램의 선형 크기와 dB 크기 비교

일반적인 STFT 는 주파수 축과 크기 축이 모두 선형이다. 반면 **청각 시스템**에는 다음 두 가지 특성이 있다.

- 주파수를 **로그 스케일**로 인지한다. 저주파 영역의 100 Hz 차이가 고주파 영역의 1 kHz 차이보다 크게 느껴진다.
- 크기도 **로그(dB) 스케일**로 인지한다.

이 특성을 모사한 것이 `librosa.feature.melspectrogram` 이다. 같은 Mel 스펙트로그램을 크기의 표현만 선형(linear)과 dB 로 바꾸어 두 장으로 그려 비교한다. `win_length=2048`, `hop_length=1024` 로 앞 실습과 같은 시간 간격(약 23 ms)을 사용한다. Mel 축은 1 kHz 아래에서 거의 선형이므로 세로축이 0 ~ 1024 Hz 인 이 그림에서는 로그 압축이 나타나지 않는다. `set_ylim` 두 줄을 지우고 전체 대역을 그리면 고주파 쪽이 압축된 Mel 축을 볼 수 있다.

In [ ]:
win_length = 2048
hop_length = 1024

S = librosa.feature.melspectrogram(y=v_f1, sr=fs_f1,
                                    win_length=win_length,
                                    hop_length=hop_length)
S_dB = librosa.power_to_db(S, ref=np.max)

fig, ax = plt.subplots(1, 2, figsize=(16, 5))

img0 = librosa.display.specshow(S, x_axis='time', y_axis='mel',
                                 sr=fs_f1, hop_length=hop_length,
                                 cmap='magma', ax=ax[0])
ax[0].set_title('Mel Spectrogram — Linear')
ax[0].set_ylim(0, 1024)
fig.colorbar(img0, ax=ax[0], label='Power (linear)')

img1 = librosa.display.specshow(S_dB, x_axis='time', y_axis='mel',
                                 sr=fs_f1, hop_length=hop_length,
                                 cmap='magma', ax=ax[1])
ax[1].set_title('Mel Spectrogram — dB (ref = max)')
ax[1].set_ylim(0, 1024)
fig.colorbar(img1, ax=ax[1], label='Power (dB)')

fig.tight_layout(); plt.show()

**관찰.** 선형 크기로 그린 왼쪽 패널에서는 크기가 가장 큰 성분이 있는 일부 셀을 제외한 대부분의 영역이 거의 검게 보이므로 구조를 읽기 어렵다. dB 로 바꾸면 같은 데이터의 **동적 범위**가 압축되어 약한 고조파 성분과 잡음 바닥까지 한 장에 나타난다. 이것이 음성과 음악처럼 사람이 듣는 소리를 시각화할 때 로그 주파수 축인 Mel 축과 dB 크기의 조합을 널리 사용하는 이유이다. 고조파의 간격 자체가 진단 정보인 기계 신호는 선형 주파수 축으로 분석한다.

---


## 요약

- STFT 는 **시간과 주파수**의 두 축에서 신호를 동시에 분석한다.
- 윈도우 길이(`nperseg`)는 **시간 분해능과 주파수 분해능**의 트레이드오프를 정하는 핵심 조절 변수이며, 엔진음에서는 `2048` 이 실용적인 절충점이다.
- 겹침(`noverlap`)은 시간 축 방향의 **시각적 부드러움**을 좌우하지만 실제 정보량은 늘리지 않으며, 표준은 50 ~ 75 %이다.
- 주파수 분해능을 우선하는 긴 윈도우는 거의 변하지 않는 고조파의 관찰에, 중간 길이 윈도우는 시간에 따라 변하는 고조파와 chirp 의 관찰에, 시간 분해능을 우선하는 짧은 윈도우는 충격 이벤트의 추적에 적합하다.
- 청각적 해석이 필요하면 **Mel 축과 dB 크기**의 조합을 사용한다.